# Análisis de Mercado de Videojuegos

Análisis estratégico de ventas históricas de videojuegos para identificar patrones de éxito por plataforma, género, región y señales de reseñas.

## 1. Configuración

Importaciones, rutas del proyecto, funciones reutilizables y configuración de visualización.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import preprocess_games, filter_relevant_period
from src.analysis import (
    annual_release_counts,
    platform_sales,
    platform_lifecycle,
    genre_sales,
    review_sales_correlation,
    regional_top,
)
from src.hypothesis import compare_user_scores

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


## 2. Carga y Preparación de Datos

El dataset se normaliza en una tabla analítica consistente. Los valores `tbd` en calificaciones de usuarios se tratan como faltantes, las clasificaciones ausentes se marcan como `Unknown` y las ventas globales se calculan a partir de las ventas regionales.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'games.csv'
raw_games = pd.read_csv(DATA_PATH)
games = preprocess_games(raw_games)
recent = filter_relevant_period(games, start_year=2012, end_year=2016)

print('Raw shape:', raw_games.shape)
print('Prepared shape:', games.shape)
print('Recent market window:', recent['year_of_release'].min(), '-', recent['year_of_release'].max())
games.head()


## 3. Revisión de Calidad de Datos

Esta tabla resume tipos de datos, valores faltantes y cardinalidad después del preprocesamiento.

In [ ]:
quality = pd.DataFrame({
    'dtype': games.dtypes.astype(str),
    'missing': games.isna().sum(),
    'unique': games.nunique(dropna=True),
})
quality


## 4. Tendencia de Lanzamientos

El volumen de lanzamientos por año ayuda a decidir qué periodos históricos siguen siendo relevantes para planeación futura.

In [ ]:
release_counts = annual_release_counts(games)

plt.figure(figsize=(12, 5))
sns.lineplot(data=release_counts, x='year_of_release', y='games_released', marker='o')
plt.title('Game releases by year')
plt.xlabel('Release year')
plt.ylabel('Games released')
plt.tight_layout()
plt.show()

release_counts.tail(10)


## 5. Estructura de Mercado por Plataforma

Las plataformas se ordenan por ventas globales para identificar líderes históricos y entender dinámicas de generaciones de consolas.

In [ ]:
platform_rank = platform_sales(games)
platform_rank.head(10)


In [ ]:
top_platforms = platform_rank.head(10)['platform']
platform_yearly = (
    games[games['platform'].isin(top_platforms)]
    .groupby(['year_of_release', 'platform'], as_index=False)['global_sales']
    .sum()
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=platform_yearly, x='year_of_release', y='global_sales', hue='platform', marker='o')
plt.title('Global sales trend for top platforms')
plt.xlabel('Release year')
plt.ylabel('Global sales, USD millions')
plt.tight_layout()
plt.show()


## 6. Ciclo de Vida de Plataformas

Una plataforma puede dominar históricamente y aun así no ser relevante para una campaña futura. Las métricas de ciclo de vida separan fuerza histórica de oportunidad actual.

In [ ]:
lifecycle = platform_lifecycle(games)
lifecycle.head(12)


## 7. Ventana de Mercado Reciente

Para planeación comercial, los años recientes contienen más señal operativa que todo el archivo histórico.

In [ ]:
recent_platform_rank = platform_sales(recent)
recent_platform_rank.head(10)


In [ ]:
recent_top = recent_platform_rank.head(8)['platform']
plt.figure(figsize=(12, 5))
sns.boxplot(data=recent[recent['platform'].isin(recent_top)], x='platform', y='global_sales', showfliers=False)
plt.title('Sales distribution by recent leading platforms')
plt.xlabel('Platform')
plt.ylabel('Global sales, USD millions')
plt.tight_layout()
plt.show()


## 8. Reseñas vs Ventas

El análisis compara la relación entre calificaciones de críticos/usuarios y ventas en plataformas seleccionadas. No prueba causalidad, pero indica si las reseñas pueden ayudar a priorizar.

In [ ]:
for platform in ['PS4', 'XOne', '3DS']:
    if platform in recent['platform'].unique():
        print(f'\n{platform}')
        print(review_sales_correlation(recent, platform).round(3))


## 9. Desempeño por Género

Las ventas por género muestran dónde se concentra la demanda y dónde la estrategia de campaña puede necesitar posicionamiento distinto.

In [ ]:
genre_rank = genre_sales(recent)
plt.figure(figsize=(12, 5))
sns.barplot(data=genre_rank, x='genre', y='global_sales')
plt.title('Recent global sales by genre')
plt.xlabel('Genre')
plt.ylabel('Global sales, USD millions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

genre_rank


## 10. Perfiles Regionales

Norteamérica, Europa y Japón se analizan por separado porque las preferencias de plataformas y géneros no son uniformes globalmente.

In [ ]:
regions = ['na_sales', 'eu_sales', 'jp_sales']
for region in regions:
    print(f'\nTop platforms by {region}')
    display(regional_top(recent, region, 'platform'))
    print(f'\nTop genres by {region}')
    display(regional_top(recent, region, 'genre'))


## 11. Perfiles por Clasificación ESRB

Las clasificaciones ESRB se comparan por región. Las categorías faltantes o desconocidas deben interpretarse con cuidado, especialmente fuera de Norteamérica.

In [ ]:
rating_profiles = {}
for region in ['na_sales', 'eu_sales', 'jp_sales']:
    rating_profiles[region] = (
        recent.groupby('rating')[region]
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )
    print(f'\n{region}')
    display(rating_profiles[region])


## 12. Pruebas de Hipótesis

Se evalúan dos comparaciones con diagnóstico de varianza mediante Levene y pruebas t independientes:

1. Calificaciones de usuarios de Xbox One vs PC.
2. Calificaciones de usuarios de Acción vs Deportes.

In [ ]:
xone_pc = compare_user_scores(recent, 'platform', 'XOne', 'PC', alpha=0.05)
action_sports = compare_user_scores(recent, 'genre', 'Action', 'Sports', alpha=0.05)

pd.DataFrame([xone_pc.__dict__, action_sports.__dict__])


## 13. Conclusiones

- El momento de la plataforma es un factor clave de relevancia comercial.
- Las plataformas de generación reciente son más útiles para planeación que las plataformas históricas.
- El desempeño por género cambia entre mercados.
- Japón requiere una lectura de mercado distinta a Norteamérica y Europa.
- Las calificaciones de críticos suelen ser más informativas comercialmente que las de usuarios.
- Las pruebas de hipótesis añaden disciplina estadística a las comparaciones de mercado.